# NB13 이더리움 가상머신과 가스 — 장난감 EVM, 거래 적용, 가스 소진

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB13-1: 장난감 EVM — 바이트코드를 한 명령씩 실행하며 가스를 깎는다 (공부용, 실사용 금지)
# 요금표 값은 황서(상하이판) 부록 G에서 그대로 옮겼다. 명령 집합과 저장소 비용 규칙은 크게 단순화했다.
G_VERYLOW, G_MID, G_HIGH, G_JUMPDEST = 3, 8, 10, 1
G_COLDSLOAD, G_WARMACCESS, G_SSET, G_SRESET = 2100, 100, 20000, 2900
MOD = 2**256

OPS = {0x00: "STOP", 0x01: "ADD", 0x02: "MUL", 0x03: "SUB", 0x54: "SLOAD", 0x55: "SSTORE",
       0x56: "JUMP", 0x57: "JUMPI", 0x5b: "JUMPDEST", 0x60: "PUSH1", 0x80: "DUP1"}

class OutOfGas(Exception):
    def __init__(self, msg, steps=0):
        super().__init__(msg); self.steps = steps      # 멈추기 전까지 실행한 명령 수를 함께 담는다

def run(code: bytes, storage: dict, gas: int, trace: bool = False):
    """code 를 0번지부터 실행한다. 성공: (남은 가스, 새 저장소, 실행한 명령 수). 가스 부족: OutOfGas.
    저장소는 복사본에서 바꾸므로, 예외가 나면 호출자가 원래 저장소를 그대로 쓰면 「되돌리기」가 된다."""
    st = dict(storage); stack = []; pc = 0; touched = set(); steps = 0
    def charge(c):
        nonlocal gas
        if gas < c:
            raise OutOfGas(f"pc={pc}: {c} 가스 필요, 남은 가스 {gas}", steps)
        gas -= c
    while pc < len(code):
        name = OPS.get(code[pc])
        if name is None:
            raise OutOfGas(f"pc={pc}: 알 수 없는 명령 0x{code[pc]:02x}")   # 실제 EVM은 별도 예외. 여기서는 단순화
        steps += 1
        if trace:
            print(f"    pc={pc:>2} {name:<8} stack={stack} gas={gas}")
        if name == "STOP":
            break
        elif name == "PUSH1":
            charge(G_VERYLOW); stack.append(code[pc + 1]); pc += 2; continue
        elif name == "DUP1":
            charge(G_VERYLOW); stack.append(stack[-1])
        elif name in ("ADD", "MUL", "SUB"):
            charge(G_VERYLOW); b, a = stack.pop(), stack.pop()
            stack.append({"ADD": (a + b) % MOD, "MUL": (a * b) % MOD, "SUB": (a - b) % MOD}[name])
        elif name == "SLOAD":
            key = stack.pop(); charge(G_WARMACCESS if key in touched else G_COLDSLOAD); touched.add(key)
            stack.append(st.get(key, 0))
        elif name == "SSTORE":
            key, val = stack.pop(), stack.pop()
            cost = 0 if key in touched else G_COLDSLOAD
            touched.add(key)
            cost += G_SSET if (st.get(key, 0) == 0 and val != 0) else G_SRESET
            charge(cost)
            st[key] = val
        elif name == "JUMPDEST":
            charge(G_JUMPDEST)
        elif name == "JUMP":
            charge(G_MID); pc = stack.pop(); continue
        elif name == "JUMPI":
            charge(G_HIGH); dest, cond = stack.pop(), stack.pop()
            if cond != 0:
                pc = dest; continue
        pc += 1
    return gas, st, steps

# OneNumber.set(42) 의 핵심 저장소 연산을 흉내 낸 바이트코드: PUSH1 42, PUSH1 0, SSTORE, STOP  (칸 0 에 42 를 쓴다; ABI 해석·RETURN 은 뺐다)
set42 = bytes([0x60, 42, 0x60, 0, 0x55, 0x00])
gas_left, st, n = run(set42, {}, gas=30_000, trace=True)
print(f"빈 저장소에 42 쓰기: 명령 {n}개, 쓴 가스 {30_000 - gas_left} (= 3 + 3 + 2100 + 20000), 저장소 {st}")

set7 = bytes([0x60, 7, 0x60, 0, 0x55, 0x00])
gas_left2, st2, n2 = run(set7, st, gas=30_000)
print(f"42 를 7 로 덮어쓰기:  명령 {n2}개, 쓴 가스 {30_000 - gas_left2} (= 3 + 3 + 2100 + 2900), 저장소 {st2}")

# OneNumber.get() 의 핵심 연산을 흉내 냄: PUSH1 0, SLOAD, STOP  — 읽기는 쓰기보다 훨씬 싸다
get = bytes([0x60, 0, 0x54, 0x00])
gas_left3, _, _ = run(get, st2, gas=30_000)
print(f"칸 0 읽기:            쓴 가스 {30_000 - gas_left3} (= 3 + 2100)")

    pc= 0 PUSH1    stack=[] gas=30000
    pc= 2 PUSH1    stack=[42] gas=29997
    pc= 4 SSTORE   stack=[42, 0] gas=29994
    pc= 5 STOP     stack=[] gas=7894
빈 저장소에 42 쓰기: 명령 4개, 쓴 가스 22106 (= 3 + 3 + 2100 + 20000), 저장소 {0: 42}
42 를 7 로 덮어쓰기:  명령 4개, 쓴 가스 5006 (= 3 + 3 + 2100 + 2900), 저장소 {0: 7}
칸 0 읽기:            쓴 가스 2103 (= 3 + 2100)


In [ ]:
# NB13-2: 거래 적용 — 유효성 검사, 선지급, 실행, 되돌리기, 환불, 기본 수수료 소각·우선 수수료 지급 (공부용)
G_TRANSACTION, G_TXDATAZERO, G_TXDATANONZERO = 21000, 4, 16

def intrinsic_gas(data: bytes) -> int:                       # 황서 식 (64) 중 보통 거래(계약 생성·접근 목록 없음) 부분
    return G_TRANSACTION + sum(G_TXDATAZERO if b == 0 else G_TXDATANONZERO for b in data)

def apply_tx(state: dict, tx: dict, base_fee: int, beneficiary: str):
    """state: {addr: {"nonce","balance","code"(bytes 또는 None),"storage"(dict)}}.
    tx: {"from","to","value","data","gas_limit","max_fee","max_priority","nonce"}.  단위는 전부 장난감 값."""
    S = {a: {**acc, "storage": dict(acc.get("storage", {}))} for a, acc in state.items()}
    sender = S[tx["from"]]
    g0 = intrinsic_gas(tx["data"])
    # --- 내재적 유효성 (황서 §6 의 (3)(5)(6)(7)(8)) ---
    if tx["nonce"] != sender["nonce"]:                  return S, f"무효: 논스 {tx['nonce']} ≠ 계정 논스 {sender['nonce']}"
    if tx["gas_limit"] < g0:                            return S, f"무효: 가스 한도 {tx['gas_limit']} < 내재 가스 {g0}"
    if tx["max_fee"] < base_fee:                        return S, f"무효: 최대 수수료 {tx['max_fee']} < 기본 수수료 {base_fee}"
    if tx["max_priority"] > tx["max_fee"]:              return S, "무효: 우선 수수료 > 최대 수수료"
    v0 = tx["gas_limit"] * tx["max_fee"] + tx["value"]
    if sender["balance"] < v0:                          return S, f"무효: 잔고 {sender['balance']} < 선지급액 {v0}"
    # --- 유효 가격 (식 (66)(67)) 과 비가역 선지급 (식 (73)(74)) ---
    priority = min(tx["max_priority"], tx["max_fee"] - base_fee)
    price = base_fee + priority
    sender["nonce"] += 1
    sender["balance"] -= tx["gas_limit"] * price
    gas = tx["gas_limit"] - g0
    # --- 실행 ---
    to = S.setdefault(tx["to"], {"nonce": 0, "balance": 0, "code": None, "storage": {}})
    status = "성공"
    try:
        if to["code"]:
            gas, new_storage, _ = run(to["code"], to["storage"], gas)
            to["storage"] = new_storage
        sender["balance"] -= tx["value"]; to["balance"] += tx["value"]
    except OutOfGas as e:
        status = f"가스 소진({e}) → 실행 되돌림"
        gas = 0                                           # 저장소·송금 변경은 버린다(원래 상태 유지). 선지급과 논스 증가는 남는다
    # --- 환불과 배분 (식 (82) 의 환불 계수기 Ar 는 이 장난감에서 0) ---
    gas_used = tx["gas_limit"] - gas
    sender["balance"] += gas * price                       # 안 쓴 가스는 산 값 그대로 돌려준다
    S.setdefault(beneficiary, {"nonce": 0, "balance": 0, "code": None, "storage": {}})["balance"] += gas_used * priority
    burned = gas_used * base_fee
    return S, f"{status} | 쓴 가스 {gas_used} | 보낸 사람 부담 {gas_used * price} = 소각 {burned} + 검증자 {gas_used * priority}"

state = {"Alice": {"nonce": 0, "balance": 10_000_000, "code": None, "storage": {}},
         "Store": {"nonce": 1, "balance": 0, "code": set42, "storage": {}}}
BASE = 10                                                # 장난감 기본 수수료(가스당)
tx1 = {"from": "Alice", "to": "Store", "value": 0, "data": b"", "gas_limit": 50_000, "max_fee": 15, "max_priority": 2, "nonce": 0}
state, msg = apply_tx(state, tx1, BASE, "Val")
print("tx1 set(42):", msg)
print("   Store.storage =", state["Store"]["storage"], "| Alice 잔고", state["Alice"]["balance"], "| Val 잔고", state["Val"]["balance"])

tx2 = {**tx1, "nonce": 1, "gas_limit": 21_500}            # 내재 가스 21000 은 넘지만 SSTORE 까지는 못 간다
state, msg = apply_tx(state, tx2, BASE, "Val")
print("tx2 가스 한도 21,500:", msg)
print("   Store.storage =", state["Store"]["storage"], "(그대로) | Alice 잔고", state["Alice"]["balance"], "(수수료는 나갔다) | 논스", state["Alice"]["nonce"])

tx3 = {**tx1, "nonce": 2, "max_fee": 9}                   # 기본 수수료 10 보다 낮게 적은 거래
_, msg = apply_tx(state, tx3, BASE, "Val")
print("tx3 최대 수수료 9:", msg)

tx1 set(42): 성공 | 쓴 가스 43106 | 보낸 사람 부담 517272 = 소각 431060 + 검증자 86212
   Store.storage = {0: 42} | Alice 잔고 9482728 | Val 잔고 86212
tx2 가스 한도 21,500: 가스 소진(pc=4: 5000 가스 필요, 남은 가스 494) → 실행 되돌림 | 쓴 가스 21500 | 보낸 사람 부담 258000 = 소각 215000 + 검증자 43000
   Store.storage = {0: 42} (그대로) | Alice 잔고 9224728 (수수료는 나갔다) | 논스 2
tx3 최대 수수료 9: 무효: 최대 수수료 9 < 기본 수수료 10


In [ ]:
# NB13-3: 무한 루프 — 멈추지 않는 코드를 가스가 멈춘다 (공부용)
# JUMPDEST(0) ; PUSH1 0 ; JUMP  → 0번지로 영원히 되돌아간다. 한 바퀴에 1 + 3 + 8 = 12 가스.
forever = bytes([0x5b, 0x60, 0x00, 0x56])
for budget in [100, 1_000, 100_000]:
    try:
        run(forever, {}, gas=budget)
        print("끝났다?!")                                  # 이 줄은 실행되지 않아야 한다
    except OutOfGas as e:
        print(f"가스 {budget:>7}: 명령 {e.steps:>6}개 실행(약 {e.steps // 3:>5}바퀴) 뒤 멈춤 — {e}")

state2 = {"Alice": {"nonce": 0, "balance": 10_000_000, "code": None, "storage": {}},
          "Loop": {"nonce": 1, "balance": 0, "code": forever, "storage": {}}}
tx = {"from": "Alice", "to": "Loop", "value": 0, "data": b"", "gas_limit": 100_000, "max_fee": 15, "max_priority": 2, "nonce": 0}
state2, msg = apply_tx(state2, tx, BASE, "Val")
print("무한 루프 계약에 거래:", msg)
print("   Alice 잔고", state2["Alice"]["balance"], "— 가스 한도 전부를 산 값(100,000 × 12 = 1,200,000)이 그대로 나갔다. 노드는 100,000 가스어치만 일했다")

가스     100: 명령     27개 실행(약     9바퀴) 뒤 멈춤 — pc=3: 8 가스 필요, 남은 가스 0
가스    1000: 명령    252개 실행(약    84바퀴) 뒤 멈춤 — pc=3: 8 가스 필요, 남은 가스 0
가스  100000: 명령  25002개 실행(약  8334바퀴) 뒤 멈춤 — pc=3: 8 가스 필요, 남은 가스 0
무한 루프 계약에 거래: 가스 소진(pc=3: 8 가스 필요, 남은 가스 0) → 실행 되돌림 | 쓴 가스 100000 | 보낸 사람 부담 1200000 = 소각 1000000 + 검증자 200000
   Alice 잔고 8800000 — 가스 한도 전부를 산 값(100,000 × 12 = 1,200,000)이 그대로 나갔다. 노드는 100,000 가스어치만 일했다
